In [4]:
import os
import shutil
import random
from pathlib import Path
import cv2
import numpy as np
import torch
import yaml
from ultralytics import YOLO
from sklearn.model_selection import train_test_split


os.environ["YOLO_OFFLINE"] = "true"
os.environ["YOLO_VERBOSE"] = "True"

# Force PyTorch to ignore MPS and CUDA, strictly using CPU.
DEVICE = "cpu"
print(f"[SYSTEM] Pipeline locked to failsafe mode: {DEVICE.upper()}")

# File paths and directories
DATASET_DIR = Path(r"/Users/akintanoreofeoluwa/Downloads/LeJEPA_YoloV8_Pretraining_Checkpoints/image_dataset")
CHECKPOINT_PATH = Path(r"/Users/akintanoreofeoluwa/Downloads/LeJEPA _pretrainining_multi_camera_boll/checkpoints_cam235_50k_dense_lejepa/dense_lejepa_yolov8_cotton_50k_checkpoint.pth")
OUTPUT_DIR = Path("./yolov8_finetune_results")
VIS_OUTPUT_DIR = OUTPUT_DIR / "visualizations_overlap"

# EMERGENCY SPEED HYPERPARAMETERS
IMAGE_SIZE = 256       # Smallest viable resolution for maximum speed on CPU
BATCH_SIZE = 16
EPOCHS = 40            
CONF_THRESH = 0.25     
MAX_DETECTIONS = 300   
VAL_RATIO = 0.20
DATASET_SUBSET_RATIO = 1.0  
RANDOM_SEED = 42
CLASS_NAMES = ["cotton_boll"]


def prepare_and_split_dataset(dataset_dir: Path, val_ratio: float = 0.20, subset_ratio: float = 1.0, seed: int = 42) -> Path:
    split_dir = dataset_dir / "split_dataset"
    
    if split_dir.exists():
        print("[DATASET] Removing stale split directory to prevent data leakage...")
        shutil.rmtree(split_dir)

    img_train_dir = split_dir / "images" / "train"
    img_val_dir = split_dir / "images" / "val"
    lbl_train_dir = split_dir / "labels" / "train"
    lbl_val_dir = split_dir / "labels" / "val"

    print(f"[DATASET] Creating structured dataset splits at: {split_dir.resolve()}")
    for p in [img_train_dir, img_val_dir, lbl_train_dir, lbl_val_dir]:
        p.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    source_img_dir = dataset_dir / "images" if (dataset_dir / "images").exists() else dataset_dir
    source_lbl_dir = dataset_dir / "labels" if (dataset_dir / "labels").exists() else dataset_dir

    raw_images = []
    for p in source_img_dir.rglob("*"):
        if p.suffix.lower() in valid_exts:
            if "train" not in p.parts and "val" not in p.parts and "split_dataset" not in p.parts:
                raw_images.append(p)

    if not raw_images:
        raise RuntimeError(f"No valid source image files found in {source_img_dir}")

    # Remove empty/background images to ensure stability
    valid_images = []
    for img_path in raw_images:
        lbl_path = source_lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists() and os.path.getsize(lbl_path) > 0:
            valid_images.append(img_path)

    print(f"[DATASET] Filtered valid labeled images: {len(valid_images)}")

    # Truncate to subset if requested
    target_size = max(1, int(len(valid_images) * subset_ratio))
    valid_images = valid_images[:target_size]
    
    # GUARANTEED STRICT SPLIT using scikit-learn
    train_images, val_images = train_test_split(valid_images, test_size=val_ratio, random_state=seed)

    print(f"[DATASET] Clean split summary: {len(train_images)} Training | {len(val_images)} Validation.")

    def copy_and_sanitize_pairs(image_list, dst_img_dir, dst_lbl_dir):
        for img_path in image_list:
            # Copy Image
            shutil.copy2(img_path, dst_img_dir / img_path.name)
            
            lbl_path = source_lbl_dir / f"{img_path.stem}.txt"
            dst_lbl_path = dst_lbl_dir / lbl_path.name
            
            # CRITICAL: Sanitize labels by forcing class ID to 0 to prevent index crashes
            with open(lbl_path, 'r') as f_in, open(dst_lbl_path, 'w') as f_out:
                for line in f_in:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # Write "0" as the class ID, followed by the original coordinates
                        f_out.write(f"0 {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n")

    copy_and_sanitize_pairs(train_images, img_train_dir, lbl_train_dir)
    copy_and_sanitize_pairs(val_images, img_val_dir, lbl_val_dir)
    print("[DATASET] Data partitioning and label sanitization completed successfully.")

    yaml_path = split_dir / "dataset.yaml"
    data_dict = {
        "path": str(split_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {0: CLASS_NAMES[0]},
    }

    with open(yaml_path, "w") as f:
        yaml.dump(data_dict, f, default_flow_style=False)

    return yaml_path


def load_pretrained_backbone(model: YOLO, checkpoint_path: Path):
    if not checkpoint_path.exists():
        print(f"[WARNING] Checkpoint does not exist: {checkpoint_path}")
        return

    print("\n================================================")
    print("[INIT] Loading LeJEPA backbone")
    print("================================================")

    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except Exception:
        import numpy._core.multiarray
        torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

    backbone_layers = {}
    for key, value in checkpoint.items():
        if not key.startswith("backbone.layers."):
            continue
        tokens = key.split(".")
        layer_idx = int(tokens[2])
        layer_param = ".".join(tokens[3:])

        if layer_idx not in backbone_layers:
            backbone_layers[layer_idx] = {}
        backbone_layers[layer_idx][layer_param] = value

    loaded_layers = 0
    for layer_idx in range(10):
        if layer_idx in backbone_layers:
            target_layer = model.model.model[layer_idx]
            target_layer.load_state_dict(backbone_layers[layer_idx], strict=False)
            loaded_layers += 1

    print(f"Backbone layers successfully loaded: {loaded_layers}/10")
    print("================================================\n")


def parse_yolo_label_file(label_path: Path, img_width: int, img_height: int) -> list:
    boxes = []
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id, xc, yc, w, h = map(float, parts)
                    x1 = int((xc - w / 2) * img_width)
                    y1 = int((yc - h / 2) * img_height)
                    x2 = int((xc + w / 2) * img_width)
                    y2 = int((yc + h / 2) * img_height)
                    boxes.append((int(cls_id), x1, y1, x2, y2))
    return boxes


def visualize_bounding_box_overlap(model: YOLO, dataset_dir: Path, output_dir: Path, conf_thresh: float = 0.25, max_images: int = 20):
    output_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir = dataset_dir / "split_dataset" / "images" / "val"
    val_lbl_dir = dataset_dir / "split_dataset" / "labels" / "val"

    img_paths = [p for p in val_img_dir.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")][:max_images]
    print(f"\n[VISUALIZATION] Generating overlays for {len(img_paths)} validation samples...")

    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w, _ = img.shape
        lbl_path = val_lbl_dir / f"{img_path.stem}.txt"
        
        for cls_id, x1, y1, x2, y2 in parse_yolo_label_file(lbl_path, w, h):
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, f"GT", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        results = model.predict(source=str(img_path), imgsz=IMAGE_SIZE, conf=conf_thresh, device=DEVICE, verbose=False)
        if results[0].boxes is not None:
            for box in results[0].boxes:
                px1, py1, px2, py2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0].cpu().item())
                cv2.rectangle(img, (px1, py1), (px2, py2), (0, 0, 255), 2)
                cv2.putText(img, f"{conf:.2f}", (px1, min(h - 5, py2 + 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

        cv2.imwrite(str(output_dir / f"overlap_{img_path.name}"), img)
    print(f"[VISUALIZATION] Saved to: {output_dir.resolve()}")


def run_finetuning():
    yaml_path = prepare_and_split_dataset(DATASET_DIR, val_ratio=VAL_RATIO, subset_ratio=DATASET_SUBSET_RATIO, seed=RANDOM_SEED)

    model = YOLO("yolov8n.pt")
    load_pretrained_backbone(model, CHECKPOINT_PATH)

    print("\n================================================")
    print(f"STAGE 1: FAILSAFE CPU FINE-TUNING ({EPOCHS} Epochs)")
    print("================================================")

    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,        # Forces CPU
        optimizer="AdamW",
        lr0=0.002,          
        lrf=0.01,
        weight_decay=1e-4,
        freeze=None,        
        cache=False,        
        close_mosaic=10,
        patience=20,
        conf=CONF_THRESH,
        max_det=MAX_DETECTIONS,
        amp=False,            # Disables mixed precision bug
        project=str(OUTPUT_DIR),
        name="finetuned_lejepa_yolov8_failsafe",
        exist_ok=True,
        workers=0,
        plots=True,
        val=True,
    )

    print("\n================================================")
    print("\n================================================")
    print("STAGE 2: FINAL EVALUATION")
    print("================================================")

    # The 'model' object is automatically updated with the best weights after model.train() completes!
    
    # Run final validation directly on the existing model object
    metrics = model.val(data=str(yaml_path), device=DEVICE, split="val", imgsz=IMAGE_SIZE, conf=CONF_THRESH)
    
    print("\nFINAL METRICS (mAP):")
    print(f"mAP@50: {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")
    
    # Pass 'model' instead of 'trained_model'
    visualize_bounding_box_overlap(model, DATASET_DIR, VIS_OUTPUT_DIR, conf_thresh=0.25, max_images=20)

if __name__ == "__main__":
    run_finetuning()

[SYSTEM] Pipeline locked to failsafe mode: CPU
[DATASET] Removing stale split directory to prevent data leakage...
[DATASET] Creating structured dataset splits at: /Users/akintanoreofeoluwa/Downloads/LeJEPA_YoloV8_Pretraining_Checkpoints/image_dataset/split_dataset
[DATASET] Filtered valid labeled images: 865
[DATASET] Clean split summary: 692 Training | 173 Validation.
[DATASET] Data partitioning and label sanitization completed successfully.

[INIT] Loading LeJEPA backbone
Backbone layers successfully loaded: 10/10


STAGE 1: FAILSAFE CPU FINE-TUNING (40 Epochs)
Ultralytics 8.4.115 🚀 Python-3.12.5 torch-2.8.0 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/akintanoreofeoluwa/Do